In [14]:
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.sql import functions as F
import time
import psutil
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.mllib.evaluation import MulticlassMetrics
import numpy as np

In [15]:
# Check data hadoop
PATH_TO_CSV = "../../hdfs/data-input/drug200.parquet"
pass_linux = 'echo 123 | sudo -S'
!{pass_linux} docker exec namenode hdfs dfs -mkdir -p /user/data/drug
!{pass_linux} docker cp "{PATH_TO_CSV}" namenode:/tmp/drug200.parquet
!{pass_linux} docker exec namenode hdfs dfs -put /tmp/drug200.parquet /user/data/drug
!{pass_linux} docker exec namenode hdfs dfs -ls -R /user/data/drug

[sudo] password for lenovo: [sudo] password for lenovo: Successfully copied 395MB to namenode:/tmp/drug200.parquet
[sudo] password for lenovo: put: `/user/data/drug/drug200.parquet': File exists
[sudo] password for lenovo: -rw-r--r--   2 root supergroup  395495382 2026-05-06 12:15 /user/data/drug/drug200.parquet


In [16]:
memory_use = "20g"
spark = (SparkSession.builder
    .appName("Optimized_RF_50M_Rows")
    # Allocate 12GB for drivers, leaving 4GB for the OS and Docker overhead.
    .config("spark.driver.memory", memory_use)
    .config("spark.executor.memory", memory_use)
    
    # Memory Optimization: Allocate 80% of RAM to Spark,
    # Prioritizing Execution (RF computation) over Storage (cache)
    .config("spark.memory.fraction", "0.8")
    .config("spark.memory.storageFraction", "0.2")
    
    # Data partitioning: 50 million rows should be divided into at least 500 partitions.
    .config("spark.sql.shuffle.partitions", "500")
    .config("spark.default.parallelism", "500")
    
    # Avoid timeouts when processing large volumes.
    .config("spark.network.timeout", "1200s")
    .config("spark.sql.broadcastTimeout", "1200s")

    .config("spark.jars.packages", "com.redislabs:spark-redis_2.12:3.1.0")

    .getOrCreate())

In [17]:
# Read from HDFS
ip_namenode = '172.20.0.2'
spark.sparkContext.setCheckpointDir(f"hdfs://{ip_namenode}:9000/user/checkpoints")
hdfs_path = f"hdfs://{ip_namenode}:9000/user/data/drug/drug200.parquet"
df = spark.read.parquet(hdfs_path, header=True, inferSchema=True)
# Check load data
df = df.repartition(500) 
df.show(5)
df.printSchema()

+------+---+---+------+-----------+-------+-----+
|row_id|Age|Sex|    BP|Cholesterol|Na_to_K| Drug|
+------+---+---+------+-----------+-------+-----+
|698434| 35|  F|   LOW|       HIGH| 21.095|DrugY|
|749267| 18|  M|   LOW|       HIGH|  30.23|DrugY|
|479356| 53|  F|   LOW|     NORMAL| 15.839|DrugY|
|745917| 57|  M|  HIGH|       HIGH|  9.448|DrugB|
|641846| 44|  F|NORMAL|     NORMAL| 27.084|DrugY|
+------+---+---+------+-----------+-------+-----+
only showing top 5 rows
root
 |-- row_id: long (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Sex: string (nullable = true)
 |-- BP: string (nullable = true)
 |-- Cholesterol: string (nullable = true)
 |-- Na_to_K: float (nullable = true)
 |-- Drug: string (nullable = true)



In [18]:
df.count()

50000000

In [19]:
# Lấy số lượng hàng (Row count)
row_count = df.count()
col_count = len(df.columns)
print(f"Kích thước tập dữ liệu: {row_count} hàng x {col_count} cột")

Kích thước tập dữ liệu: 50000000 hàng x 7 cột


In [22]:
df['Age','Na_to_K'].describe().show()

+-------+------------------+------------------+
|summary|               Age|           Na_to_K|
+-------+------------------+------------------+
|  count|          50000000|          50000000|
|   mean|       44.50098632|20.499214869852036|
| stddev|17.318983621047654| 8.370598377268237|
|    min|                15|               6.0|
|    max|                74|              35.0|
+-------+------------------+------------------+

